In [1]:
import os, sqlite3

In [ ]:
DB_PATH = "experiment.db"

def fresh(pk_decl):
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)
    conn = sqlite3.connect(DB_PATH, autocommit=False)
    conn.execute(f"CREATE TABLE clients (id {pk_decl}, name TEXT NOT NULL)")
    conn.commit()
    return conn

def rows(conn):
    return conn.execute("SELECT id, name FROM clients ORDER BY id").fetchall()

# 'sqlite_sequence' is an internal SQLite table that exists only when some table uses AUTOINCREMENT, it records the highest ID ever issued
def seq(conn):
    try: 
        return conn.execute("SELECT name, seq FROM sqlite_sequence").fetchall()
    except sqlite3.OperationalError:
        return "no sqlite_sequence table"

## without AUTOINCREMENT

In [8]:
PK_DECL = "INTEGER PRIMARY KEY"

conn = fresh(PK_DECL)
conn.execute("INSERT INTO clients (name) VALUES (?)", ("Farm A",))
conn.commit()
print("rows: ", rows(conn), " | sequence: ", seq(conn))

rows:  [(1, 'Farm A')]  | sequence:  no sqlite_sequence table


In [9]:
cur = conn.execute("INSERT INTO clients (name) VALUES (?)", ("Farm B",))
print("Farm B was assigned id: ", cur.lastrowid)
print("before rollback: ", rows(conn), " | sequence: ", seq(conn))

conn.rollback()
print("after rollback: ", rows(conn), " | sequence: ", seq(conn))

conn.close()
conn = sqlite3.connect(DB_PATH, autocommit=False)
print("after reopen: ", rows(conn))

Farm B was assigned id:  2
before rollback:  [(1, 'Farm A'), (2, 'Farm B')]  | sequence:  no sqlite_sequence table
after rollback:  [(1, 'Farm A')]  | sequence:  no sqlite_sequence table
after reopen:  [(1, 'Farm A')]


In [10]:
conn.execute("INSERT INTO clients (name) VALUES (?)", ("Farm C",))
conn.commit()
print("rows: ", rows(conn), " | sequence: ", seq(conn))

rows:  [(1, 'Farm A'), (2, 'Farm C')]  | sequence:  no sqlite_sequence table


In [11]:
max_id = conn.execute("SELECT MAX(id) FROM clients").fetchone()[0]
conn.execute("DELETE FROM clients WHERE id = ?", (max_id,))
conn.commit()
print("deleted id", max_id, "-> rows: ", rows(conn), " | sequence: ", seq(conn))

conn.execute("INSERT INTO clients (name) VALUES (?)", ("Farm D",))
conn.commit()
print("after inserting Farm D: ", rows(conn), " | sequence: ", seq(conn))

deleted id 2 -> rows:  [(1, 'Farm A')]  | sequence:  no sqlite_sequence table
after inserting Farm D:  [(1, 'Farm A'), (2, 'Farm D')]  | sequence:  no sqlite_sequence table


## with AUTOINCREMENT

In [12]:
PK_DECL = "INTEGER PRIMARY KEY AUTOINCREMENT"

conn = fresh(PK_DECL)
conn.execute("INSERT INTO clients (name) VALUES (?)", ("Farm A",))
conn.commit()
print("rows: ", rows(conn), " | sequence: ", seq(conn))

rows:  [(1, 'Farm A')]  | sequence:  [('clients', 1)]


In [13]:
cur = conn.execute("INSERT INTO clients (name) VALUES (?)", ("Farm B",))
print("Farm B was assigned id: ", cur.lastrowid)
print("before rollback: ", rows(conn), " | sequence: ", seq(conn))

conn.rollback()
print("after rollback: ", rows(conn), " | sequence: ", seq(conn))

conn.close()
conn = sqlite3.connect(DB_PATH, autocommit=False)
print("after reopen: ", rows(conn))

Farm B was assigned id:  2
before rollback:  [(1, 'Farm A'), (2, 'Farm B')]  | sequence:  [('clients', 2)]
after rollback:  [(1, 'Farm A')]  | sequence:  [('clients', 1)]
after reopen:  [(1, 'Farm A')]


In [14]:
conn.execute("INSERT INTO clients (name) VALUES (?)", ("Farm C",))
conn.commit()
print("rows: ", rows(conn), " | sequence: ", seq(conn))

rows:  [(1, 'Farm A'), (2, 'Farm C')]  | sequence:  [('clients', 2)]


In [15]:
max_id = conn.execute("SELECT MAX(id) FROM clients").fetchone()[0]
conn.execute("DELETE FROM clients WHERE id = ?", (max_id,))
conn.commit()
print("deleted id", max_id, "-> rows: ", rows(conn), " | sequence: ", seq(conn))

conn.execute("INSERT INTO clients (name) VALUES (?)", ("Farm D",))
conn.commit()
print("after inserting Farm D: ", rows(conn), " | sequence: ", seq(conn))

deleted id 2 -> rows:  [(1, 'Farm A')]  | sequence:  [('clients', 2)]
after inserting Farm D:  [(1, 'Farm A'), (3, 'Farm D')]  | sequence:  [('clients', 3)]
